In [1]:
import numpy as np
import pandas as pd
from copy import deepcopy

# Grid dimensions
Grid_Size = 10
Start = (0, 0)
Goal = (9, 9)

# Action definitions
Actions = {0: 'Up', 1: 'Down', 2: 'Left', 3: 'Right'}
Action_Deltas = {
    0: (-1, 0),  # Up
    1: (1, 0),   # Down
    2: (0, -1),  # Left
    3: (0, 1)    # Right
}

print("Environment initialized: 10x10 grid")
print(f"Start: {Start}, Goal: {Goal}")
print(f"Actions: {list(Actions.values())}")

Environment initialized: 10x10 grid
Start: (0, 0), Goal: (9, 9)
Actions: ['Up', 'Down', 'Left', 'Right']


In [2]:
def get_valid_actions(state):
    i, j = state
    valid = []
    if i > 0: valid.append(0)              # Up
    if i < Grid_Size - 1: valid.append(1)  # Down
    if j > 0: valid.append(2)              # Left
    if j < Grid_Size - 1: valid.append(3)  # Right
    return valid

def get_next_states(state, action):
    next_states = {}
    i, j = state
    
    if state == Goal:
        return {GoalL: 1.0}
    
    valid_actions = get_valid_actions(state)
    
    # Intended action (80%)
    di, dj = Action_Deltas[action]
    intended_next = (i + di, j + dj)
    next_states[intended_next] = next_states.get(intended_next, 0) + 0.8
    
    # Random action (20% split equally among other valid actions)
    other_actions = [a for a in valid_actions if a != action]
    for other_action in other_actions:
        di, dj = Action_Deltas[other_action]
        random_next = (i + di, j + dj)
        next_states[random_next] = next_states.get(random_next, 0) + 0.2 / len(other_actions)
    
    return next_states

def get_reward(state, action):
    if state == Goal:
        return 0  # Terminal state
    return -1

In [3]:
def value_iteration(gamma, threshold=1e-4, max_iterations=1000):
    # Initialize value function
    V = {}
    for i in range(Grid_Size):
        for j in range(Grid_Size):
            V[(i, j)] = 0.0
    
    policy = {}
    iterations = 0
    
    for iteration in range(max_iterations):
        delta = 0
        V_new = deepcopy(V)
        
        # Update each state
        for i in range(Grid_Size):
            for j in range(Grid_Size):
                state = (i, j)
                
                if state == Goal:
                    V_new[state] = 10  # Goal state value
                    continue
                
                valid_actions = get_valid_actions(state)
                action_values = []
                
                # Calculate Q-value for each valid action
                for action in valid_actions:
                    next_states = get_next_states(state, action)
                    reward = get_reward(state, action)
                    
                    # Q(s,a) = R(s,a) + γ * Σ P(s'|s,a) * V(s')
                    q_value = reward
                    for next_state, prob in next_states.items():
                        q_value += gamma * prob * V[next_state]
                    
                    action_values.append((q_value, action))
                
                # Update value with best action
                best_value = max(action_values, key=lambda x: x[0])[0]
                delta = max(delta, abs(best_value - V[state]))
                V_new[state] = best_value
        
        V = V_new
        iterations += 1
        
        # Check convergence
        if delta < threshold:
            print(f"Converged in {iterations} iterations (γ={gamma})")
            break
    
    # Extract optimal policy
    for i in range(Grid_Size):
        for j in range(Grid_Size):
            state = (i, j)
            if state == Goal:
                policy[state] = 'Goal'
                continue
            
            valid_actions = get_valid_actions(state)
            best_action = None
            best_value = -float('inf')
            
            for action in valid_actions:
                next_states = get_next_states(state, action)
                reward = get_reward(state, action)
                q_value = reward + gamma * sum(prob * V[next_state] 
                                               for next_state, prob in next_states.items())
                if q_value > best_value:
                    best_value = q_value
                    best_action = action
            
            policy[state] = Actions[best_action]
    
    return V, policy, iterations

In [4]:
gamma_09 = 0.9
V_09, policy_09, iters_09 = value_iteration(gamma_09)

print(f"\nValue Iteration Results: (γ = {gamma_09})")

# Display value function as grid
print("\nOptimal Value Function:")
V_grid = np.zeros((Grid_Size, Grid_Size))
for i in range(Grid_Size):
    for j in range(Grid_Size):
        V_grid[i, j] = V_09[(i, j)]

print(np.round(V_grid, 2))

# Display policy as grid
print("\nOptimal Policy:")
policy_grid = np.empty((Grid_Size,Grid_Size), dtype=object)
for i in range(Grid_Size):
    for j in range(Grid_Size):
        policy_grid[i, j] = policy_09[(i, j)]

for row in policy_grid:
    print(' '.join(f"{action:>6}" for action in row))

Converged in 44 iterations (γ=0.9)

Value Iteration Results: (γ = 0.9)

Optimal Value Function:
[[-8.23 -8.04 -7.76 -7.45 -7.09 -6.69 -6.23 -5.71 -5.21 -4.96]
 [-8.04 -7.83 -7.52 -7.16 -6.75 -6.27 -5.72 -5.11 -4.51 -4.2 ]
 [-7.76 -7.52 -7.17 -6.75 -6.27 -5.72 -5.09 -4.38 -3.66 -3.26]
 [-7.45 -7.16 -6.75 -6.27 -5.72 -5.09 -4.37 -3.54 -2.69 -2.15]
 [-7.09 -6.75 -6.27 -5.72 -5.09 -4.36 -3.53 -2.58 -1.56 -0.86]
 [-6.69 -6.27 -5.72 -5.09 -4.36 -3.53 -2.57 -1.48 -0.27  0.66]
 [-6.23 -5.72 -5.09 -4.37 -3.53 -2.57 -1.47 -0.21  1.21  2.45]
 [-5.71 -5.11 -4.38 -3.54 -2.58 -1.48 -0.21  1.24  2.91  4.56]
 [-5.21 -4.51 -3.66 -2.69 -1.56 -0.27  1.21  2.91  4.84  7.05]
 [-4.96 -4.2  -3.26 -2.15 -0.86  0.66  2.45  4.56  7.05 10.  ]]

Optimal Policy:
  Down  Right  Right  Right  Right  Right  Right   Down   Down   Down
  Down   Down  Right  Right  Right  Right   Down   Down   Down   Down
  Down   Down   Down  Right  Right   Down   Down   Down   Down   Down
  Down   Down   Down   Down   Down   Down   Do

In [5]:
# γ = 0 (myopic/greedy)
print("Running Value Iteration for γ = 0 (Myopic Agent)...")
V_0, policy_0, iters_0 = value_iteration(gamma=0, threshold=1e-4)

print("\nValue Function (γ = 0):")
V_grid_0 = np.zeros((Grid_Size,Grid_Size))
for i in range(Grid_Size):
    for j in range(Grid_Size):
        V_grid_0[i, j] = V_0[(i, j)]
print(np.round(V_grid_0, 2))

Running Value Iteration for γ = 0 (Myopic Agent)...
Converged in 2 iterations (γ=0)

Value Function (γ = 0):
[[-1. -1. -1. -1. -1. -1. -1. -1. -1. -1.]
 [-1. -1. -1. -1. -1. -1. -1. -1. -1. -1.]
 [-1. -1. -1. -1. -1. -1. -1. -1. -1. -1.]
 [-1. -1. -1. -1. -1. -1. -1. -1. -1. -1.]
 [-1. -1. -1. -1. -1. -1. -1. -1. -1. -1.]
 [-1. -1. -1. -1. -1. -1. -1. -1. -1. -1.]
 [-1. -1. -1. -1. -1. -1. -1. -1. -1. -1.]
 [-1. -1. -1. -1. -1. -1. -1. -1. -1. -1.]
 [-1. -1. -1. -1. -1. -1. -1. -1. -1. -1.]
 [-1. -1. -1. -1. -1. -1. -1. -1. -1. 10.]]


In [6]:
# γ = 1 (future-oriented)
print("Running Value Iteration for γ = 1 (No Discounting)...")
V_1, policy_1, iters_1 = value_iteration(gamma=1.0, threshold=1e-4)

print("\nValue Function (γ = 1):")
V_grid_1 = np.zeros((Grid_Size,Grid_Size))
for i in range(Grid_Size):
    for j in range(Grid_Size):
        V_grid_1[i, j] = V_1[(i, j)]
print(np.round(V_grid_1, 2))

Running Value Iteration for γ = 1 (No Discounting)...
Converged in 56 iterations (γ=1.0)

Value Function (γ = 1):
[[-13.87 -12.87 -11.61 -10.32  -9.04  -7.76  -6.49  -5.24  -4.16  -3.72]
 [-12.87 -11.93 -10.63  -9.3   -7.96  -6.62  -5.28  -3.97  -2.84  -2.36]
 [-11.61 -10.63  -9.35  -8.    -6.64  -5.28  -3.92  -2.59  -1.42  -0.87]
 [-10.32  -9.3   -8.    -6.64  -5.28  -3.92  -2.56  -1.22  -0.     0.63]
 [ -9.04  -7.96  -6.64  -5.28  -3.92  -2.56  -1.19   0.16   1.42   2.14]
 [ -7.76  -6.62  -5.28  -3.92  -2.56  -1.19   0.17   1.53   2.83   3.68]
 [ -6.49  -5.28  -3.92  -2.56  -1.19   0.17   1.54   2.9    4.23   5.22]
 [ -5.24  -3.97  -2.59  -1.22   0.16   1.53   2.9    4.27   5.63   6.79]
 [ -4.16  -2.84  -1.42  -0.     1.42   2.83   4.23   5.63   7.01   8.38]
 [ -3.72  -2.36  -0.87   0.63   2.14   3.68   5.22   6.79   8.38  10.  ]]


In [7]:
print("OBSERVATIONS & ANALYSIS\n")


print("Convergence Iterations:")
print(f"\tγ = 0.0 : {iters_0} iterations")
print(f"\tγ = 0.9 : {iters_09} iterations")
print(f"\tγ = 1.0 : {iters_1} iterations")

print("\nValue at Start State (0,0):")
print(f"\tγ = 0.0 : {V_0[(0,0)]:.2f}")
print(f"\tγ = 0.9 : {V_09[(0,0)]:.2f}")
print(f"\tγ = 1.0 : {V_1[(0,0)]:.2f}")

print("\nKey Insights:")
print("""
1. γ = 0 (Myopic):
   - Only immediate rewards matter
   - Value ≈ -1 everywhere (one step cost)
   - Converges quickly
   - Policies may be suboptimal for long-term goals

2. γ = 0.9 (Standard):
   - Balances immediate and future rewards
   - Values reflect optimal path to goal
   - More realistic discounting
   - Takes moderate iterations to converge

3. γ = 1.0 (No Discounting):
   - All future rewards weighted equally
   - Values can become very negative (long paths)
   - May not converge with undiscounted rewards
   - Prioritizes any path to goal equally
""")

# Show policy difference for start position and neighbors
print("\nPolicy Comparison at Key Positions:")
positions = [(0,0), (0,1), (1,0), (5,5), (8,8), (9,9)]
data = []
for pos in positions:
    data.append({
        'State': str(pos),
        'γ=0': policy_0[pos],
        'γ=0.9': policy_09[pos],
        'γ=1': policy_1[pos],
        'Value(0.9)': f"{V_09[pos]:.2f}"
    })

df = pd.DataFrame(data)
print(df.to_string(index=False))

OBSERVATIONS & ANALYSIS

Convergence Iterations:
	γ = 0.0 : 2 iterations
	γ = 0.9 : 44 iterations
	γ = 1.0 : 56 iterations

Value at Start State (0,0):
	γ = 0.0 : -1.00
	γ = 0.9 : -8.23
	γ = 1.0 : -13.87

Key Insights:

1. γ = 0 (Myopic):
   - Only immediate rewards matter
   - Value ≈ -1 everywhere (one step cost)
   - Converges quickly
   - Policies may be suboptimal for long-term goals

2. γ = 0.9 (Standard):
   - Balances immediate and future rewards
   - Values reflect optimal path to goal
   - More realistic discounting
   - Takes moderate iterations to converge

3. γ = 1.0 (No Discounting):
   - All future rewards weighted equally
   - Values can become very negative (long paths)
   - May not converge with undiscounted rewards
   - Prioritizes any path to goal equally


Policy Comparison at Key Positions:
 State  γ=0 γ=0.9   γ=1 Value(0.9)
(0, 0) Down  Down  Down      -8.23
(0, 1) Down Right Right      -8.04
(1, 0)   Up  Down  Down      -8.04
(5, 5)   Up  Down  Down      -3.53
(